<a href="https://colab.research.google.com/github/kadreatharv04/APS/blob/main/Lab_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MATRIX CHAIN MULTIPLICATION Dynamic Programming with table filling.
Problem Description:
Find minimum number of scalar multiplications needed to multiply a chain of matrices..

#Theory
Cost of multiplying two matrices:
If A is p×q and B is q×r, cost = p × q × r

Dynamic Programming idea:
Break problem into smaller subproblems.
Store results in a table to avoid recomputation.

m[i][j] = minimum cost to multiply Ai to Aj

## Algorithm
1. Take array p[] of size n
2. Create table m[n][n]
3. Set m[i][i] = 0
4. Fill table diagonally
5. For each chain length L:
    Compute m[i][j]
    Try all possible k between i and j
6. Choose minimum cost
7. Return m[1][n-1]

In [ ]:
import sys

def matrix_chain_multiplication(p):
    n = len(p)

    # Create DP table for minimum costs
    m = [[0 for _ in range(n)] for _ in range(n)]
    # Create DP table for split points (to reconstruct optimal parenthesization)
    s = [[0 for _ in range(n)] for _ in range(n)]

    # L is chain length
    for L in range(2, n): # L from 2 to n-1
        for i in range(1, n - L + 1): # i from 1 to n-L
            j = i + L - 1 # j from i+L-1 to n-1
            m[i][j] = sys.maxsize # Initialize with a very large value

            # Try all possible split points k
            for k in range(i, j): # k from i to j-1
                # Cost of multiplying Ai..Ak and Ak+1..Aj, plus cost of multiplying results
                cost = (m[i][k] +
                        m[k+1][j] +
                        p[i-1] * p[k] * p[j])

                if cost < m[i][j]:
                    m[i][j] = cost
                    s[i][j] = k # Store the split point

    return m, s # Return both tables

In [ ]:
# Input dimensions
p = [10, 30, 5, 60]

table, s_table = matrix_chain_multiplication(p)

print("Minimum multiplication cost:", table[1][len(p)-1])

print("\nDP Table (costs):")
for row in table[1:]:
    print(row[1:])

print("\nDP Table (split points):")
for row in s_table[1:]:
    print(row[1:])

Minimum multiplication cost: 4500

DP Table (costs):
[0, 1500, 4500]
[0, 0, 9000]
[0, 0, 0]

DP Table (split points):
[0, 1, 2]
[0, 0, 2]
[0, 0, 0]


In [ ]:
def print_optimal_parenthesization(s, i, j):
    if i == j:
        return f"A{i}"
    else:
        k = s[i][j]
        left = print_optimal_parenthesization(s, i, k)
        right = print_optimal_parenthesization(s, k + 1, j)
        return f"({left} x {right})"

def calculate_naive_cost(p):
    # For a chain of n matrices (p has n+1 elements),
    # a naive multiplication would be to multiply them in order: ((A1 x A2) x A3) ...
    # The cost would be p[0]*p[1]*p[2] + p[0]*p[2]*p[3] + ...
    # This is not strictly a single 'naive' cost as there are many, but we can
    # calculate a simple sequential multiplication.
    # Let's consider a simple sequential multiplication: A1(A2(...(An-1 An)...))
    # This is equivalent to ((A1 A2) A3) ... An.
    n = len(p) - 1 # Number of matrices
    if n <= 1:
        return 0

    cost = 0
    # For (A1 x A2) x A3 x ... An
    # Cost of A1 x A2 is p[0]*p[1]*p[2]
    # Resulting matrix is p[0] x p[2]
    # Cost of (A1xA2) x A3 is p[0]*p[2]*p[3]
    # ...
    # Sum of products for A[i] x A[i+1] is sum(p[i-1]*p[i]*p[i+1]) for i=1 to n-1
    # A more illustrative naive: (A1 x A2) x (A3 x A4)...
    # A straightforward naive sequential multiplication like (((A1A2)A3)A4)...An
    # This would be (p[0]*p[1]*p[2]) + (p[0]*p[2]*p[3]) + ...
    # A simple bound is multiplying matrices in sequence A1 * (A2 * (A3 * ... (An-1 * An)))
    # Or ((((A1 * A2) * A3) * A4) ... * An)
    # The cost of (A_1 * A_2) is p_0*p_1*p_2
    # The cost of ((A_1 * A_2) * A_3) is p_0*p_1*p_2 + p_0*p_2*p_3
    # The cost of (((A_1 * A_2) * A_3) * A_4) is p_0*p_1*p_2 + p_0*p_2*p_3 + p_0*p_3*p_4
    # So, the naive cost can be approximated by summing p[0] * p[k] * p[k+1] for k from 1 to n-1
    naive_cost = 0
    for k in range(1, n):
        naive_cost += p[0] * p[k] * p[k+1]
    return naive_cost


# Exercise: p = [5, 10, 3, 12, 5, 50, 6]
p_exercise = [5, 10, 3, 12, 5, 50, 6]
n_exercise = len(p_exercise)

m_exercise, s_exercise = matrix_chain_multiplication(p_exercise)

min_cost_exercise = m_exercise[1][n_exercise - 1]
print(f"Exercise Problem: p = {p_exercise}")
print(f"Minimum multiplication cost: {min_cost_exercise}")

print("\nOptimal Parenthesization:")
print(print_optimal_parenthesization(s_exercise, 1, n_exercise - 1))

# Calculate and compare with a naive multiplication cost
# A naive approach, for example, multiplying in a fixed order like (A1 * A2) * (A3 * A4) ...
# Or, more simply, just multiplying in sequence from left to right: (((A1 * A2) * A3) * ... * An)
# Cost of multiplying A_i-1 x A_i (dimensions p_i-1 x p_i) and A_i x A_i+1 (dimensions p_i x p_i+1) is p_i-1 * p_i * p_i+1
# A simple baseline is if we multiply all matrices in left-to-right order:
# (A1 A2) A3 A4 ... An => cost = p[0]*p[1]*p[2]
# ( (A1 A2) A3 ) A4 ... An => cost = p[0]*p[1]*p[2] + p[0]*p[2]*p[3]
# This assumes the first dimension p[0] persists.

# A simpler naive cost can be to just sum up all pairwise multiplications in sequence
# e.g., for A1, A2, A3, A4 (dims p0,p1,p2,p3,p4)
# Cost of (A1A2) is p0*p1*p2
# Cost of (A2A3) is p1*p2*p3
# Cost of (A3A4) is p2*p3*p4
# A crude naive total would be the sum of all A_i * A_i+1 products
# Or, a simplified naive calculation where we just do A1x(A2x(A3x...))
# or (((A1xA2)xA3)xA4)
# For ((A1xA2)xA3)xA4 the cost is p[0]p[1]p[2] + p[0]p[2]p[3] + p[0]p[3]p[4]

# Let's use the provided calculate_naive_cost, which computes a left-to-right sequential multiplication
naive_cost = calculate_naive_cost(p_exercise)
print(f"Naive multiplication cost (sequential left-to-right): {naive_cost}")

saved_multiplications = naive_cost - min_cost_exercise
print(f"Multiplications saved: {saved_multiplications}")


Exercise Problem: p = [5, 10, 3, 12, 5, 50, 6]
Minimum multiplication cost: 2010

Optimal Parenthesization:
((A1 x A2) x ((A3 x A4) x (A5 x A6)))
Naive multiplication cost (sequential left-to-right): 3380
Multiplications saved: 1370


# Exercise
1. Given p = [5, 10, 3, 12, 5, 50, 6]
2. Compute minimum cost
3. Print optimal parenthesization
4. Compare number of multiplications saved